In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_TOTAL = True
REUSE_SINUSES = True
RUN_HIGHRES_CHAMBERS_IF_LICENSED = True
TOTALSEG_LICENSE = ""  # optional; leave blank to skip licensed heartchambers_highres
OUT_ROOT = "/content/drive/MyDrive/OpenPlaque/GPU_Whole_Heart_Context_v1"
SAVE_FULL_TOTAL_OUTPUT = True


# OpenPlaque — GPU whole-heart context segmentation

Runs a full TotalSegmentator `total` model on the frozen source CCTA, plus the dedicated `aortic_sinuses` model. If a TotalSegmentator academic/non-commercial license is supplied above, it also runs `heartchambers_highres`.

Purpose: create anatomical context for coronary/plaque rendering — heart, aorta, root/cusps, and optionally myocardium/chambers. Research use only.

In [ ]:
!pip -q install TotalSegmentator SimpleITK nibabel pandas matplotlib
import os, sys, json, shutil, subprocess, zipfile
from pathlib import Path
import numpy as np, pandas as pd, SimpleITK as sitk, matplotlib.pyplot as plt

repo=Path("/content/OpenPlaque")
if repo.exists(): shutil.rmtree(repo)
!git clone -q --depth 1 --branch whole-heart-context-segmentation-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!nvidia-smi
!TotalSegmentator --version
!totalseg_info --classes -ta aortic_sinuses


In [ ]:
# Reconstruct a geometry-correct source CCTA NIfTI from the frozen cache.
cache = Path("/content/drive/MyDrive/OpenPlaque/Cache/Secondary_3D_Vesselness_Topology_v1")
arr_path = cache/"series7_int16.npy"
meta_path = cache/"series7_int16.json"
if not arr_path.exists() or not meta_path.exists():
    raise FileNotFoundError("Missing frozen source-CCTA cache")

meta = json.loads(meta_path.read_text())
arr = np.load(arr_path, mmap_mode="r")
img = sitk.GetImageFromArray(arr)
spacing_zyx = np.asarray(meta["spacing_zyx"], float)
img.SetSpacing(tuple(spacing_zyx[::-1]))
origin = tuple(np.asarray(meta["positions_lps_mm"][0], float))
img.SetOrigin(origin)

iop = np.asarray(meta["image_orientation_patient"], float)
row = iop[:3]; col = iop[3:]
slc = np.cross(row, col)
direction = np.array([[row[0], col[0], slc[0]],
                      [row[1], col[1], slc[1]],
                      [row[2], col[2], slc[2]]], float)
img.SetDirection(tuple(direction.ravel()))
source_nii = Path("/content/source_ccta.nii.gz")
sitk.WriteImage(img, str(source_nii))
print(source_nii, arr.shape, img.GetSpacing(), img.GetOrigin(), img.GetDirection())


In [ ]:
out_root=Path(OUT_ROOT); out_root.mkdir(parents=True,exist_ok=True)

def run_task(task,name,reuse,license_number=None,preview=False):
    out=out_root/name
    done=out/"_SUCCESS"
    if reuse and done.exists():
        print("reuse",task,out); return out
    if out.exists(): shutil.rmtree(out)
    out.mkdir(parents=True)
    cmd=["TotalSegmentator","-i",str(source_nii),"-o",str(out),"-ta",task,"--device","gpu"]
    if preview: cmd += ["--preview"]
    if license_number: cmd += ["-l",license_number]
    print("RUN:", " ".join(cmd))
    subprocess.run(cmd,check=True)
    done.write_text("done")
    return out

total_dir=run_task("total","total",REUSE_TOTAL,preview=True)
sinus_dir=run_task("aortic_sinuses","aortic_sinuses",REUSE_SINUSES)

chamber_dir=None
if RUN_HIGHRES_CHAMBERS_IF_LICENSED and TOTALSEG_LICENSE.strip():
    chamber_dir=run_task("heartchambers_highres","heartchambers_highres",False,license_number=TOTALSEG_LICENSE.strip())
else:
    print("Skipping licensed heartchambers_highres (no license supplied).")


In [ ]:
# Summarize context masks and select key structures.
def mask_stats(folder):
    rows=[]
    for p in sorted(Path(folder).glob("*.nii.gz")):
        try:
            im=sitk.ReadImage(str(p)); a=sitk.GetArrayFromImage(im)>0
            rows.append({"source":Path(folder).name,"structure":p.name.replace(".nii.gz",""),
                         "voxels":int(a.sum()),"volume_ml":float(a.sum()*np.prod(im.GetSpacing())/1000.0)})
        except Exception as e:
            print("skip",p,e)
    return rows

rows=mask_stats(total_dir)+mask_stats(sinus_dir)
if chamber_dir: rows += mask_stats(chamber_dir)
stats=pd.DataFrame(rows).sort_values(["source","volume_ml"],ascending=[True,False])
stats.to_csv(out_root/"whole_heart_structure_volumes.csv",index=False)
display(stats.head(80))

# Copy selected masks to a compact context folder.
selected=out_root/"selected_context_masks"; selected.mkdir(exist_ok=True)
wanted = {
    "heart","aorta","left_ventricular_outflow_tract",
    "right_coronary_cusp","left_coronary_cusp","non_coronary_cusp",
    "heart_myocardium","heart_atrium_left","heart_ventricle_left","heart_atrium_right","heart_ventricle_right",
    "myocardium","atrium_left","ventricle_left","atrium_right","ventricle_right","pulmonary_artery"
}
for folder in [total_dir,sinus_dir]+([chamber_dir] if chamber_dir else []):
    for p in Path(folder).glob("*.nii.gz"):
        if p.name.replace(".nii.gz","") in wanted:
            shutil.copy2(p,selected/f"{Path(folder).name}__{p.name}")


In [ ]:
# Axial context montage using heart/aorta/root masks that exist.
src=sitk.GetArrayFromImage(sitk.ReadImage(str(source_nii)))
mask_paths=list(selected.glob("*.nii.gz"))
loaded=[]
for p in mask_paths:
    im=sitk.ReadImage(str(p)); a=sitk.GetArrayFromImage(im)>0
    if a.shape==src.shape: loaded.append((p.stem.replace(".nii",""),a))
if not loaded:
    raise RuntimeError("No selected context masks matched source geometry")

combined=np.zeros(src.shape,dtype=np.uint8)
for _,a in loaded: combined |= a.astype(np.uint8)
score=combined.sum(axis=(1,2)); ids=np.argsort(score)[-9:][::-1]
fig,axes=plt.subplots(3,3,figsize=(14,14))
for ax,z in zip(axes.ravel(),ids):
    ax.imshow(src[z],cmap="gray",vmin=-200,vmax=900)
    ax.imshow(combined[z],alpha=.30)
    ax.set_title(f"context z={z}"); ax.axis("off")
fig.tight_layout(); fig.savefig(out_root/"whole_heart_context_qc.png",dpi=180); plt.show()

preview=total_dir/"preview.png"
if preview.exists():
    shutil.copy2(preview,out_root/"totalsegmentator_preview.png")


In [ ]:
summary={
    "status":"COMPLETE",
    "tasks_run":["total","aortic_sinuses"]+(["heartchambers_highres"] if chamber_dir else []),
    "highres_chambers_run":bool(chamber_dir),
    "selected_mask_count":len(list(selected.glob("*.nii.gz"))),
    "purpose":"anatomical context for coronary tree, plaque, PCAT, and aortic-root visualizations"
}
(out_root/"summary.json").write_text(json.dumps(summary,indent=2))
html=out_root/"OPENPLAQUE_GPU_WHOLE_HEART_CONTEXT_REPORT.html"
html.write_text("<html><body><h1>OpenPlaque GPU whole-heart context</h1><p>Research use only.</p>"+
                stats.head(100).to_html(index=False)+
                "<p><img src='whole_heart_context_qc.png' style='max-width:100%'></p>"+
                ("<p><img src='totalsegmentator_preview.png' style='max-width:100%'></p>" if (out_root/"totalsegmentator_preview.png").exists() else "")+
                "</body></html>")
zip_path=out_root/"OPENPLAQUE_GPU_WHOLE_HEART_CONTEXT_REPORT_BACK.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in [html,out_root/"summary.json",out_root/"whole_heart_structure_volumes.csv",out_root/"whole_heart_context_qc.png"]:
        if p.exists(): z.write(p,p.name)
    if (out_root/"totalsegmentator_preview.png").exists(): z.write(out_root/"totalsegmentator_preview.png","totalsegmentator_preview.png")
    for p in selected.glob("*.nii.gz"): z.write(p,"selected_context_masks/"+p.name)
print("FINAL:",zip_path)
